[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/99_isolated_stem_final_summary.ipynb)


# 99 - Isolated-Stem Final Summary

## Research question

Which completed isolated-stem experiments should be reported as the final baseline, ablation findings, and selected MERT configuration?

## Approach

This notebook reads the registry and result folders only. It does not train models or extract embeddings. Selection should use validation macro-F1; held-out test metrics are displayed for reporting after that selection.

## What is fixed

Unless a selected config says otherwise: MedleyDB instrument labels, the `largest_balanced` split, MERT-v1-95M, 5 s segments, validation-based model selection, and test metrics saved only after training.

## What is varied

Registry filters and the selected experiment group for report tables.

## Expected interpretation

Use this notebook to produce concise tables and point to saved confusion matrices, classification reports, and predictions for qualitative error analysis.


## References

This notebook follows the project protocol in `docs/`, the MedleyDB multitrack dataset paper by Bittner et al., the MERT music representation paper by Li et al., and standard validation practice from statistical learning and deep-learning methodology: keep the data split fixed while changing one experimental factor, select models using validation metrics, and report held-out test metrics only after selection.


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- `EXPERIMENT_GROUPS`, `DATASET_GROUPS`, `MIXTURE_GROUPS`, and `SELECTED_EXPERIMENTS` to run one config at a time.
- `REPLACE_EXISTING = True` only when intentionally overwriting a completed run.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)

SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RESULTS_DIR = RUN_ROOT / "results"
CHECKPOINTS_DIR = RUN_ROOT / "checkpoints"
METADATA_DIR = RUN_ROOT / "data" / "metadata"
CACHE_ROOT = RUN_ROOT / "data" / "cache"
REPORTS_DIR = RUN_ROOT / "data" / "reports"
for path in [RESULTS_DIR, CHECKPOINTS_DIR, METADATA_DIR, CACHE_ROOT, REPORTS_DIR, RUN_ROOT / "configs"]:
    path.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)


In [ ]:
registry_path = RESULTS_DIR / "experiment_registry.csv"
if not registry_path.exists():
    raise FileNotFoundError(f"No experiment registry found at {registry_path}. Run experiments first.")
registry = pd.read_csv(registry_path)
display(registry.tail())


In [ ]:
summary_cols = [
    "experiment_id", "subset_profile", "label_granularity", "approach", "model_name",
    "mert_layer", "pooling", "representation_mode", "classifier_type",
    "best_val_macro_f1", "test_macro_f1", "test_accuracy", "results_dir", "notes",
]
summary_cols = [column for column in summary_cols if column in registry.columns]
final_rows = registry[(registry["subset_profile"] == "largest_balanced") & (registry["label_granularity"] == "medleydb_instrument")]
if final_rows.empty:
    print("No final largest-balanced MedleyDB instrument rows yet.")
else:
    display(final_rows[summary_cols].sort_values("best_val_macro_f1", ascending=False, na_position="last"))
